In [14]:
from langgraph.graph import StateGraph , START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated
from dotenv import load_dotenv 
from pydantic import BaseModel, Field
import os
import operator


In [2]:
load_dotenv()

True

In [5]:
model= ChatOpenAI(api_key=os.getenv("LANGCHAIN_API_KEY"), model="gpt-4o-mini")

In [ ]:
class EvaluationSchema(BaseModel):
    feedback:str=Field(description="Feedback on the essay")
    score:int=Field(description="Score out of 50")
    


In [7]:
structured_model= model.with_structured_output(EvaluationSchema)


In [8]:
essay="""Artificial Intelligence (AI) has become one of the most transformative technologies of the 21st century. It is rapidly changing the way humans interact with machines and how decisions are made across industries. From healthcare to finance, AI is playing a crucial role in improving efficiency, accuracy, and productivity.

One of the major advantages of AI is its ability to process large amounts of data quickly. In healthcare, AI systems can analyze medical records and detect diseases at an early stage, which helps in saving lives. Similarly, in finance, AI algorithms are used for fraud detection and risk management, reducing human error and increasing reliability.

However, the rise of AI also brings several challenges. One of the biggest concerns is job displacement. As machines become more capable of performing tasks traditionally done by humans, many jobs may become obsolete. Additionally, there are ethical concerns regarding privacy, bias in algorithms, and decision-making transparency.

Despite these challenges, AI has the potential to create new opportunities. It can lead to the development of new industries and job roles that require advanced technical skills. Governments and organizations must focus on reskilling and upskilling the workforce to adapt to this technological shift.

In conclusion, Artificial Intelligence is a powerful tool that can significantly benefit society if used responsibly. While it presents certain risks, proper regulation and ethical practices can ensure that AI contributes positively to human progress.
"""

In [ ]:
prompt=f"""You are an expert English evaluator.

Evaluate the following essay based on:
1. Grammar (10 marks)
2. Clarity (10 marks)
3. Structure (10 marks)
4. Vocabulary (10 marks)
5. Overall Quality (10 marks)

Provide:
- Score in each category
- Total score out of 50
- Detailed feedback (strengths + improvements)
- 3 specific suggestions to improve the essay

Essay:
{essay}
"""

In [12]:
structured_model.invoke(prompt).feedback

'Strengths:\n- The essay presents a clear argument on both the benefits and challenges of Artificial Intelligence, demonstrating a good understanding of the topic.\n- The structure is logical, with a clear introduction, body paragraphs addressing different aspects, and a conclusion summarizing the main points.\n- Vocabulary is appropriate and varied, helping to articulate the ideas effectively.\n\nImprovements:\n- A few grammatical errors are present, and sentence structures can be improved for better flow and sophistication.\n- Clarity could be enhanced in some areas, as certain sentences are slightly verbose and could be more concise.\n- More specific examples or data to support claims would strengthen the argument further, particularly in discussing the challenges of AI.\n\nSuggestions for Improvement:\n1. Review the essay for grammatical errors, particularly in verb tenses and subject-verb agreement, to enhance grammatical accuracy.\n2. Aim for clearer and more concise phrasing by 

In [17]:
class UPSCState(TypedDict):
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    individual_scores:Annotated[list[int], operator.add]
    avg_score:float

In [ ]:
def evaluate_language(state:UPSCState):

    prompt=f"""You are an expert English evaluator.
    Evaluate the language  quality of the following essay based on:

1. Grammar (10 marks)
2. Clarity (10 marks)
3. Structure (10 marks)
4. Vocabulary (10 marks)
5. Overall Quality (10 marks)

Provide:
- Score in each category
- Total score out of 50
- Detailed feedback (strengths + improvements)
- 3 specific suggestions to improve the essay

Essay:
{essay}
"""
    
    output= structured_model.invoke(prompt)
    return {"language_feedback":output.feedback, "individual_scores":[output.score] }


In [22]:
def evaluate_analysis(state:UPSCState):

    prompt=f"""You are an expert English evaluator.
    Evaluate the depth of  analysis  of the following essay based on:

1. Grammar (10 marks)
2. Clarity (10 marks)
3. Structure (10 marks)
4. Vocabulary (10 marks)
5. Overall Quality (10 marks)

Provide:
- Score in each category
- Total score out of 50
- Detailed feedback (strengths + improvements)
- 3 specific suggestions to improve the essay

Essay:
{essay}
"""
    
    output= structured_model.invoke(prompt)
    return {"analysis_feedback":output.feedback, "individual_scores":[output.score] }


In [26]:
def evaluate_thought(state:UPSCState):

    prompt=f"""You are an expert English evaluator.
    Evaluate the clarity of the thought of the following essay based on:

1. Grammar (10 marks)
2. Clarity (10 marks)
3. Structure (10 marks)
4. Vocabulary (10 marks)
5. Overall Quality (10 marks)

Provide:
- Score in each category
- Total score out of 50
- Detailed feedback (strengths + improvements)
- 3 specific suggestions to improve the essay

Essay:
{essay}
"""
    
    output= structured_model.invoke(prompt)
    return {"clarity_feedback":output.feedback, "individual_scores":[output.score] }


In [27]:
def final_evaluate(state:UPSCState):

    prompt = """
    You are a senior evaluator.

    You have received evaluation results from multiple expert agents:

    1. Language Evaluation:
    {language_feedback}

    2. Analytical Evaluation:
    {analysis_feedback}

    3. Thought Evaluation:
    {thought_feedback}

    Your task is to:

    1. Combine all evaluations into a final assessment
    2. Provide:
    - Final Score out of 100
    - Overall Feedback (clear and concise)
    - Key Strengths (3 points)
    - Areas of Improvement (3 points)
    - Final Verdict (Excellent / Good / Average / Poor)

    Make sure:
    - Be consistent with previous evaluations
    - Do not contradict earlier feedback
    - Keep response structured and professional
    """
        
    overall_feedback=model.invoke(prompt).content
    avg_score = sum(state["individual_scores"]) / len(state["individual_scores"])
    return {"overall_feedback": overall_feedback, "avg_score": avg_score}


In [31]:
graph= StateGraph(UPSCState)


graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_thought", evaluate_thought)
graph.add_node("final_evaluate", final_evaluate)

graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_analysis")
graph.add_edge(START, "evaluate_thought")
graph.add_edge("evaluate_language", "final_evaluate")
graph.add_edge("evaluate_analysis", "final_evaluate")
graph.add_edge("evaluate_thought", "final_evaluate")


workflow=graph.compile()

In [32]:
initial_state={
    "essay":essay
}

workflow.invoke(initial_state)

{'essay': 'Artificial Intelligence (AI) has become one of the most transformative technologies of the 21st century. It is rapidly changing the way humans interact with machines and how decisions are made across industries. From healthcare to finance, AI is playing a crucial role in improving efficiency, accuracy, and productivity.\n\nOne of the major advantages of AI is its ability to process large amounts of data quickly. In healthcare, AI systems can analyze medical records and detect diseases at an early stage, which helps in saving lives. Similarly, in finance, AI algorithms are used for fraud detection and risk management, reducing human error and increasing reliability.\n\nHowever, the rise of AI also brings several challenges. One of the biggest concerns is job displacement. As machines become more capable of performing tasks traditionally done by humans, many jobs may become obsolete. Additionally, there are ethical concerns regarding privacy, bias in algorithms, and decision-m